In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_squared_error, mean_absolute_error
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import numpy as np

auto_mpg = fetch_ucirepo(id=9)
data = pd.concat([auto_mpg.data.features, auto_mpg.data.targets], axis=1).dropna()

X = data[['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year', 'origin']]
y = data['mpg']

scaler = MinMaxScaler()
X[['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration']] = scaler.fit_transform(
    X[['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration']]
)

years = torch.tensor(X['model_year'].values, dtype=torch.float32)
boundaries = torch.tensor([73, 76, 79])
X['year_bucket'] = torch.bucketize(years, boundaries, right=False)
X = X.drop('model_year', axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = pd.get_dummies(X_train, columns=['origin'], prefix='origin')
X_test = pd.get_dummies(X_test, columns=['origin'], prefix='origin')

X_train = X_train.astype(float)
X_test = X_test.astype(float)

X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

def set_random_seed(seed=42):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_random_seed(43)

baseline_model = nn.Sequential(
    nn.Linear(X_train_tensor.shape[1], 8),
    nn.ReLU(),
    nn.Linear(8, 4),
    nn.ReLU(),
    nn.Linear(4, 1)
)

criterion = nn.MSELoss()
optimizer = optim.SGD(baseline_model.parameters(), lr=0.001)
epochs = 200

for epoch in range(epochs):
    outputs = baseline_model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f"Эпоха [{epoch+1}/{epochs}] — Потери (Loss): {loss.item():.4f}")

with torch.no_grad():
    y_pred = baseline_model(X_test_tensor)
    mse = mean_squared_error(y_test_tensor, y_pred)
    mae = mean_absolute_error(y_test_tensor, y_pred)
    print(f"\nОшибка на тестовой выборке:")
    print(f"MSE (среднеквадратичная ошибка): {mse:.4f}")
    print(f"MAE (средняя абсолютная ошибка): {mae:.4f}")


Ошибка на тестовой выборке:
MSE (среднеквадратичная ошибка): 14.5795
MAE (средняя абсолютная ошибка): 2.9447


C:\Users\CifronPro\AppData\Local\Temp\ipykernel_13928\2137856009.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration']] = scaler.fit_transform(


In [129]:
set_random_seed(48)

model_2 = nn.Sequential(
    nn.Linear(X_train_tensor.shape[1], 8),
    nn.ReLU(),
    nn.Linear(8, 4),
    nn.ReLU(),
    nn.Linear(4, 1)
)

criterion = nn.MSELoss()
optimizer = optim.SGD(model_2.parameters(), lr=0.001)
epochs = 1000

for epoch in range(epochs):
    outputs = model_2(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # if (epoch + 1) % 20 == 0:
    #     print(f"Эпоха [{epoch+1}/{epochs}] — Потери (Loss): {loss.item():.4f}")

with torch.no_grad():
    y_pred = model_2(X_test_tensor)
    mse = mean_squared_error(y_test_tensor, y_pred)
    mae = mean_absolute_error(y_test_tensor, y_pred)
    print(f"\nОшибка на тестовой выборке:")
    print(f"MSE (среднеквадратичная ошибка): {mse:.4f}")
    print(f"MAE (средняя абсолютная ошибка): {mae:.4f}")


Ошибка на тестовой выборке:
MSE (среднеквадратичная ошибка): 6.6654
MAE (средняя абсолютная ошибка): 1.9041


Вывод: ошибок стало меньше с добавлением эпох.

In [88]:
set_random_seed(44)

model_3_1 = nn.Sequential(
    nn.Linear(X_train_tensor.shape[1], 8),
    nn.ReLU(),
    nn.Linear(8, 4),
    nn.ReLU(),
    nn.Linear(4, 2),  
    nn.ReLU(),         
    nn.Linear(2, 1)    
)

criterion = nn.MSELoss()
optimizer = optim.SGD(model_3_1.parameters(), lr=0.001)
epochs = 200

for epoch in range(epochs):
    outputs = model_3_1(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # if (epoch + 1) % 20 == 0:
    #     print(f"Эпоха [{epoch+1}/{epochs}] — Потери (Loss): {loss.item():.4f}")

with torch.no_grad():
    y_pred = model_3_1(X_test_tensor)
    mse = mean_squared_error(y_test_tensor, y_pred)
    mae = mean_absolute_error(y_test_tensor, y_pred)
    print(f"\nОшибка на тестовой выборке:")
    print(f"MSE (среднеквадратичная ошибка): {mse:.4f}")
    print(f"MAE (средняя абсолютная ошибка): {mae:.4f}")


Ошибка на тестовой выборке:
MSE (среднеквадратичная ошибка): 10.5635
MAE (средняя абсолютная ошибка): 2.4454


Вывод: результат лучше, чем при таком же количестве эпох у baseline.

In [ ]:

model_3_2 = nn.Sequential(
    nn.Linear(X_train_tensor.shape[1], 8),
    nn.ReLU(),
    nn.Linear(8, 4),
    nn.ReLU(),
    nn.Linear(4, 2),  
    nn.ReLU(),         
    nn.Linear(2, 1)    
)

criterion = nn.MSELoss()
optimizer = optim.SGD(model_3_1.parameters(), lr=0.001)
epochs = 1000

for epoch in range(epochs):
    outputs = model_3_1(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # if (epoch + 1) % 20 == 0:
    #     print(f"Эпоха [{epoch+1}/{epochs}] — Потери (Loss): {loss.item():.4f}")

with torch.no_grad():
    y_pred = model_3_1(X_test_tensor)
    mse = mean_squared_error(y_test_tensor, y_pred)
    mae = mean_absolute_error(y_test_tensor, y_pred)
    print(f"\nОшибка на тестовой выборке:")
    print(f"MSE (среднеквадратичная ошибка): {mse:.4f}")
    print(f"MAE (средняя абсолютная ошибка): {mae:.4f}")


Ошибка на тестовой выборке:
MSE (среднеквадратичная ошибка): 5.9372
MAE (средняя абсолютная ошибка): 1.7588


Вывод: результат лучше, чем при 200 эпохах, но разница не настолько большая, как у baseline.

In [164]:
set_random_seed(42)

model_4_1 = nn.Sequential(
    nn.Linear(X_train_tensor.shape[1], 32),  
    nn.ReLU(),
    nn.Linear(32, 16),                        
    nn.ReLU(),
    nn.Linear(16, 1)                         
)

criterion = nn.MSELoss()
optimizer = optim.SGD(model_3_1.parameters(), lr=0.001)
epochs = 200

for epoch in range(epochs):
    outputs = model_3_1(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # if (epoch + 1) % 20 == 0:
    #     print(f"Эпоха [{epoch+1}/{epochs}] — Потери (Loss): {loss.item():.4f}")

with torch.no_grad():
    y_pred = model_3_1(X_test_tensor)
    mse = mean_squared_error(y_test_tensor, y_pred)
    mae = mean_absolute_error(y_test_tensor, y_pred)
    print(f"\nОшибка на тестовой выборке:")
    print(f"MSE (среднеквадратичная ошибка): {mse:.4f}")
    print(f"MAE (средняя абсолютная ошибка): {mae:.4f}")


Ошибка на тестовой выборке:
MSE (среднеквадратичная ошибка): 5.5035
MAE (средняя абсолютная ошибка): 1.6816


Вывод: результаты лучше, чем у model2 при том же количестве эпох. 

In [168]:
set_random_seed(42)

model_4_1 = nn.Sequential(
    nn.Linear(X_train_tensor.shape[1], 32),  
    nn.ReLU(),
    nn.Linear(32, 16),                        
    nn.ReLU(),
    nn.Linear(16, 1)                         
)

criterion = nn.MSELoss()
optimizer = optim.SGD(model_3_1.parameters(), lr=0.001)
epochs = 1000

for epoch in range(epochs):
    outputs = model_3_1(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # if (epoch + 1) % 20 == 0:
    #     print(f"Эпоха [{epoch+1}/{epochs}] — Потери (Loss): {loss.item():.4f}")

with torch.no_grad():
    y_pred = model_3_1(X_test_tensor)
    mse = mean_squared_error(y_test_tensor, y_pred)
    mae = mean_absolute_error(y_test_tensor, y_pred)
    print(f"\nОшибка на тестовой выборке:")
    print(f"MSE (среднеквадратичная ошибка): {mse:.4f}")
    print(f"MAE (средняя абсолютная ошибка): {mae:.4f}")


Ошибка на тестовой выборке:
MSE (среднеквадратичная ошибка): 5.8190
MAE (средняя абсолютная ошибка): 1.7224


Вывод: результат примерно такой же, что и при 200 эпохах, даже слегка хуже.